In [ ]:
from pyaedt import Q3d
from pyaedt.generic.constants import PLANE
from ansys.aedt.core.q3d import Q3d

In [12]:
my_path = r"D:\Users\QDG\Nimisha\interdigital_capacitor_tests\test3\testCell0.aedt"
q3d = Q3d(specified_version="2024.1", non_graphical=False, close_on_exit=True)
q3d.modeler.model_units = "um"

PyAEDT WARNING: Argument `specified_version` is deprecated for method `__init__`; use `version` instead.
PyAEDT INFO: Python version 3.8.8 (default, Apr 13 2021, 15:08:03) [MSC v.1916 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.3.
PyAEDT INFO: Returning found Desktop session with PID 20736!
PyAEDT INFO: No project is defined. Project Project27 exists and has been read.
PyAEDT INFO: Active Design set to Q3D Extractor_9PS
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec


In [13]:
q3d["cap_width"] = str(10) + "um"
q3d["cap_gap"] = str(6) + "um"
q3d["finger_length"] = str(20) + "um"
q3d["finger_count"] = str(5)
q3d["cap_distance"] = str(50) + "um"
q3d["silicon_vacuum_thickness"] = str(675) + "um"
q3d["thickness"] = str(0.2) + "um"

In [14]:
q3d["total_inner_width"] = "cap_width * finger_count + cap_gap * (finger_count - 1)"
q3d["total_inner_length"] = "2 * cap_width + finger_length + cap_gap"
q3d["total_width"] = "2 * (total_inner_width)"

In [32]:
### OUTER BODY STUFF

outer_body = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-total_width/2", "-total_inner_length/2 - cap_distance"],
    ["total_width", "total_inner_length + 2 * cap_distance"]
)

outer_etch = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-total_inner_width/2 - cap_gap", "-total_inner_length/2 - cap_gap"],
    ["total_inner_width + 2 * cap_gap", "total_inner_length + 2 * cap_gap"])

north_knob_etch = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-cap_width/2 - cap_gap", "total_inner_length/2 + cap_gap"],
    ["cap_width + 2 * cap_gap", "cap_distance - cap_gap"]
)

south_knob_etch = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-cap_width/2 - cap_gap", "-total_inner_length/2 - cap_gap"],
    ["cap_width + 2 * cap_gap", "-cap_distance + cap_gap"]
)

total_outer_etch = q3d.modeler.unite([outer_etch, north_knob_etch, south_knob_etch])
q3d.modeler.subtract(outer_body, total_outer_etch, True)

PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 3 objects has been executed.


True

In [33]:
### INSIDE STUFF

north_knob = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-cap_width/2", "total_inner_length/2"],
    ["cap_width", "cap_distance"]
)

north_flat = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-total_inner_width/2", "total_inner_length/2"],
    ["total_inner_width", "-cap_width"]
)

south_flat = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-total_inner_width/2", "-total_inner_length/2"],
    ["total_inner_width", "cap_width"]
)

south_knob = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-cap_width/2", "-total_inner_length/2"],
    ["cap_width", "-cap_distance"]
)

inner_piece = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-total_inner_width/2", "-total_inner_length/2"],
    ["total_inner_width", "total_inner_length"]
)

q3d.modeler.copy(north_knob)
north_knob_dup = q3d.modeler.paste()[0]
q3d.modeler.copy(south_knob)
south_knob_dup = q3d.modeler.paste()[0]

q3d.modeler.subtract(total_outer_etch, [inner_piece, north_knob_dup, south_knob_dup], False)

north_body = q3d.modeler.unite([north_knob, north_flat])
south_body = q3d.modeler.unite([south_knob, south_flat])

PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.


In [35]:
# creating the etching

etching_piece = q3d.modeler.create_rectangle(
    PLANE.XY,
    ["-total_inner_width/2", "-total_inner_length/2 + cap_width"],
    ["total_inner_width", "finger_length + cap_gap"])


In [37]:
# fingers
fingers_south = []
fingers_north = []

for i in range(int(q3d.evaluate_expression("finger_count"))):
    if i % 2 == 0:
        # south fingers created
        print(q3d.evaluate_expression("-total_inner_width/2 + (cap_width * 2 + cap_gap * 2) * (" + str(i) + "/2)"))
        curr_finger_south = q3d.modeler.create_rectangle(
            PLANE.XY,
            ["-total_inner_width/2 + (cap_width * 2 + cap_gap * 2) * (" + str(i) + "/2)", "-total_inner_length/2 + cap_width"],
            ["cap_width", "finger_length"]
        )
        q3d.modeler.copy(curr_finger_south)
        finger_south_dupe = q3d.modeler.paste()[0]
        south_body = q3d.modeler.unite([south_body, finger_south_dupe])

        fingers_south.append(curr_finger_south)
    else:
        # north fingers created
        print(q3d.evaluate_expression("-total_inner_width/2 + cap_width + cap_gap + (cap_width * 2 + cap_gap * 2) * ((" + str(i) + "- 1) / 2)"))
        curr_finger_north = q3d.modeler.create_rectangle(
            PLANE.XY,
            ["-total_inner_width/2 + cap_width + cap_gap + (cap_width * 2 + cap_gap * 2) * ((" + str(i) + "- 1) / 2)", "total_inner_length/2 - cap_width"],
            ["cap_width", "-finger_length"]
        )
        q3d.modeler.copy(curr_finger_north)
        finger_north_dupe = q3d.modeler.paste()[0]
        north_body = q3d.modeler.unite([north_body, finger_north_dupe])

        fingers_north.append(curr_finger_north)
        

-3.7e-05
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.
-2.1e-05
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.
-5e-06
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.
1.1e-05
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
P

In [38]:
q3d.modeler.subtract(etching_piece, fingers_north, False)
q3d.modeler.subtract(etching_piece, fingers_south, False)

q3d.modeler.thicken_sheet(total_outer_etch, "thickness", both_sides=True)
q3d.modeler.thicken_sheet(north_body, "thickness", both_sides=True)
q3d.modeler.thicken_sheet(south_body, "thickness", both_sides=True)
q3d.modeler.thicken_sheet(etching_piece, "thickness", both_sides=True)
q3d.modeler.thicken_sheet(outer_body, "thickness", both_sides=True)

q3d.assign_material(north_body, "perfect conductor")
q3d.assign_material(south_body, "perfect conductor")
q3d.assign_material(outer_body, "perfect conductor")
q3d.assign_material(etching_piece, "vacuum")
q3d.assign_material(total_outer_etch, "vacuum")

q3d.assign_net(north_body, "North")
q3d.assign_net(south_body, "South")
q3d.assign_net(outer_body, "Ground")

setup = q3d.create_setup()
q3d.save_project(my_path)

PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Project testCell0 Saved correctly


True